📜 build_tl_datasets.py

In [1]:
import os
import rdflib
import random
from urllib.parse import urlparse
from collections import defaultdict
from PIL import Image
from sklearn.model_selection import train_test_split

# Set project root (one level above notebooks/) and ensure src/ is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)

In [ ]:
# ===== CONFIG =====
PROJECT_ROOT = "."
OUTPUT_ROOT = "data/datasets"
SPLIT_RATIO = 0.2  # validation split

# ===== KG FILES =====
SCHEMA_PATH = "ontology/schema.ttl"
BDD_TTL_PATH = "out/bdd_sample_1000.ttl"

# ===== CREATE DIRS =====
YOLO_IMG_TRAIN = os.path.join(OUTPUT_ROOT, "yolo_tl/images/train")
YOLO_IMG_VAL = os.path.join(OUTPUT_ROOT, "yolo_tl/images/val")

YOLO_LABEL_TRAIN = os.path.join(OUTPUT_ROOT, "yolo_tl/labels/train")
YOLO_LABEL_VAL = os.path.join(OUTPUT_ROOT, "yolo_tl/labels/val")

COLOR_ROOT = os.path.join(OUTPUT_ROOT, "tl_color")

In [3]:
os.path.exists(SCHEMA_PATH), os.path.exists(BDD_TTL_PATH)

(True, True)

In [4]:
for p in [
    YOLO_IMG_TRAIN, YOLO_IMG_VAL,
    YOLO_LABEL_TRAIN, YOLO_LABEL_VAL,
]:
    os.makedirs(p, exist_ok=True)

for split in ["train", "val"]:
    for c in ["red", "yellow", "green"]:
        os.makedirs(os.path.join(COLOR_ROOT, split, c), exist_ok=True)

In [5]:
# ===== LOAD GRAPH =====
g = rdflib.Graph()
g.parse(SCHEMA_PATH, format="turtle")
g.parse(BDD_TTL_PATH, format="turtle")

print(f"Loaded graph with {len(g)} triples")

Loaded graph with 188424 triples


In [6]:
# ===== SPARQL QUERY =====
query = """
PREFIX cv: <http://vision.semkg.org/onto/v0.1/>

SELECT ?img ?file ?box ?x ?y ?w ?h ?color
WHERE {
  ?img a cv:Image ;
       cv:filePath ?file ;
       cv:hasAnnotation ?ann .

  ?ann a cv:ObjectDetectionAnnotation ;
       cv:category "traffic light" ;
       cv:hasBox ?box ;
       cv:annotatesObject ?tl .

  ?box cv:x ?x ;
       cv:y ?y ;
       cv:width ?w ;
       cv:height ?h .

  ?tl cv:hasColor ?color .

  FILTER(?color IN (cv:RedColor, cv:YellowColor, cv:GreenColor))
}
"""

results = g.query(query)

In [7]:
# ===== GROUP BY IMAGE =====
image_data = defaultdict(list)

for row in results:
    image_path = str(row.file)
    x = float(row.x)
    y = float(row.y)
    w = float(row.w)
    h = float(row.h)
    color = str(row.color).split("#")[-1].replace("Color", "").lower()

    image_data[image_path].append((x, y, w, h, color))

print(f"Total images: {len(image_data)}")

Total images: 527


In [8]:
# ===== TRAIN/VAL SPLIT =====
image_paths = list(image_data.keys())
train_imgs, val_imgs = train_test_split(image_paths, test_size=SPLIT_RATIO, random_state=42)

In [9]:
# ===== PROCESS EACH IMAGE =====
def process_split(image_list, split):
    for img_path in image_list:
        full_path = os.path.join(PROJECT_ROOT, img_path)
        if not os.path.exists(full_path):
            continue

        image = Image.open(full_path).convert("RGB")
        W, H = image.size

        img_name = os.path.basename(img_path)

        if split == "train":
            img_out_dir = YOLO_IMG_TRAIN
            label_out_dir = YOLO_LABEL_TRAIN
        else:
            img_out_dir = YOLO_IMG_VAL
            label_out_dir = YOLO_LABEL_VAL

        image.save(os.path.join(img_out_dir, img_name))

        label_lines = []

        for i, (x, y, w, h, color) in enumerate(image_data[img_path]):
            # ===== YOLO FORMAT =====
            x_center = (x + w/2) / W
            y_center = (y + h/2) / H
            w_norm = w / W
            h_norm = h / H

            # class id = 0 (traffic_light)
            label_lines.append(f"0 {x_center} {y_center} {w_norm} {h_norm}")

            # ===== SAVE CROP FOR COLOR CLASSIFIER =====
            crop = image.crop((x, y, x+w, y+h))
            crop_name = f"{img_name.replace('.jpg','')}_{i}.jpg"
            color = urlparse(str(color)).path.split("/")[-1]
            crop.save(os.path.join(COLOR_ROOT, split, color, crop_name))

        # Save YOLO label file
        label_file = img_name.replace(".jpg", ".txt")
        with open(os.path.join(label_out_dir, label_file), "w") as f:
            f.write("\n".join(label_lines))


print("Processing train split...")
process_split(train_imgs, "train")

print("Processing val split...")
process_split(val_imgs, "val")

Processing train split...
Processing val split...


In [ ]:
os.path.exists("data/datasets\\tl_color\\train\\")

True

In [11]:
# ===== CREATE YOLO YAML =====
yaml_content = f"""
path: {os.path.abspath(os.path.join(OUTPUT_ROOT, "yolo_tl"))}
train: images/train
val: images/val

names:
  0: traffic_light
"""

with open(os.path.join(OUTPUT_ROOT, "yolo_tl/data.yaml"), "w") as f:
    f.write(yaml_content)

print("Dataset creation complete.")

Dataset creation complete.


In [ ]:
os.path.exists("data/datasets\\yolo_tl")

True

!yolo detect train data=data/datasets/yolo_tl/data.yaml model=yolov8n.pt epochs=50 imgsz=640

📜 train_tl_color_classifier.py

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime

In [ ]:
# =========================
# CONFIG
# =========================
DATA_ROOT = "data/datasets/tl_color"
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-4
IMG_SIZE = 224
MODEL_DIR = "artifacts/models"
FIG_DIR = "artifacts/figures"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
# =========================
# TRANSFORMS
# =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [7]:
# =========================
# DATASETS
# =========================
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_ROOT, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
print("Classes:", class_names)

Classes: ['green', 'red', 'yellow']


In [8]:
# =========================
# MODEL
# =========================
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

c:\Users\kaleem\Learn\FUB\WiSe_2025_26\Knowledge Graphs and AI Apps\kgnai\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\kaleem\Learn\FUB\WiSe_2025_26\Knowledge Graphs and AI Apps\kgnai\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
# =========================
# TRAINING LOOP
# =========================
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total

    # =========================
    # VALIDATION
    # =========================
    model.eval()
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = val_correct / val_total

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {running_loss:.4f}")
    print(f"Train Acc: {train_acc:.4f}")
    print(f"Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_path = os.path.join(
            MODEL_DIR,
            f"tl_color_best_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pt"
        )
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names
        }, best_model_path)
        print(f"Saved best model to {best_model_path}")

Epoch 1/5: 100%|██████████| 47/47 [03:58<00:00,  5.08s/it]



Epoch 1
Train Loss: 10.6662
Train Acc: 0.9312
Val Acc: 0.9347
Saved best model to models\tl_color_best_20260223_231413.pt


Epoch 2/5: 100%|██████████| 47/47 [04:40<00:00,  5.97s/it]



Epoch 2
Train Loss: 6.2221
Train Acc: 0.9546
Val Acc: 0.9496
Saved best model to models\tl_color_best_20260223_231908.pt


Epoch 3/5: 100%|██████████| 47/47 [03:30<00:00,  4.47s/it]



Epoch 3
Train Loss: 4.4242
Train Acc: 0.9726
Val Acc: 0.9258


Epoch 4/5: 100%|██████████| 47/47 [03:14<00:00,  4.14s/it]



Epoch 4
Train Loss: 3.0280
Train Acc: 0.9760
Val Acc: 0.9318


Epoch 5/5: 100%|██████████| 47/47 [03:05<00:00,  3.95s/it]



Epoch 5
Train Loss: 2.0277
Train Acc: 0.9853
Val Acc: 0.9228


In [12]:
# =========================
# CONFUSION MATRIX
# =========================
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Traffic Light Color Confusion Matrix")
plt.tight_layout()

cm_path = os.path.join(FIG_DIR, "tl_color_confusion_matrix.png")
plt.savefig(cm_path)
plt.close()

print(f"Saved confusion matrix to {cm_path}")
print("Training complete.")

Saved confusion matrix to figures\tl_color_confusion_matrix.png
Training complete.


inference_pipeline.py

In [13]:
import os
import torch
import torch.nn as nn
from torchvision import transforms, models
from ultralytics import YOLO
from PIL import Image
import numpy as np

In [ ]:
os.path.exists("artifacts/runs/detect/train7/weights/best.pt")

True

In [ ]:
# =========================
# CONFIG
# =========================
YOLO_MODEL_PATH = "artifacts/runs/detect/train7/weights/best.pt"
COLOR_MODEL_PATH = "artifacts/models/tl_color_best_20260223_231908.pt"
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VALID_TL_COLORS = ["red", "yellow", "green"]

In [19]:
# =========================
# LOAD YOLO
# =========================
detector = YOLO(YOLO_MODEL_PATH)

# =========================
# LOAD COLOR CLASSIFIER
# =========================
checkpoint = torch.load(COLOR_MODEL_PATH, map_location=DEVICE)

color_model = models.resnet18(pretrained=False)
color_model.fc = nn.Linear(color_model.fc.in_features, 3)
color_model.load_state_dict(checkpoint["model_state_dict"])
color_model = color_model.to(DEVICE)
color_model.eval()

c:\Users\kaleem\Learn\FUB\WiSe_2025_26\Knowledge Graphs and AI Apps\kgnai\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\kaleem\Learn\FUB\WiSe_2025_26\Knowledge Graphs and AI Apps\kgnai\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [20]:
class_names = checkpoint["class_names"]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [21]:
# =========================
# RANKING LOGIC
# =========================
def traffic_light_score(box, img_w, img_h):
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    area = (x2 - x1) * (y2 - y1)

    cx_n = cx / img_w
    cy_n = cy / img_h
    area_n = area / (img_w * img_h)

    center_score = 1 - abs(cx_n - 0.5)
    vertical_score = 1 - cy_n
    size_score = min(area_n * 50, 1.0)

    return 0.5 * center_score + 0.3 * vertical_score + 0.2 * size_score


# =========================
# MAIN INFERENCE FUNCTION
# =========================
def infer_image(image_path):

    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size

    # ===== DETECTION =====
    results = detector(image_path)[0]

    if len(results.boxes) == 0:
        print("No traffic lights detected.")
        return None

    detections = []

    for box in results.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box

        # ===== CROP =====
        crop = image.crop((x1, y1, x2, y2))
        input_tensor = transform(crop).unsqueeze(0).to(DEVICE)

        # ===== COLOR CLASSIFICATION =====
        with torch.no_grad():
            output = color_model(input_tensor)
            _, pred = torch.max(output, 1)

        predicted_color = class_names[pred.item()]

        # ===== SCORING =====
        score = traffic_light_score((x1, y1, x2, y2), img_w, img_h)

        detections.append({
            "box": (x1, y1, x2, y2),
            "color": predicted_color,
            "score": score
        })

    # ===== SELECT RELEVANT TL =====
    detections = sorted(detections, key=lambda x: x["score"], reverse=True)
    best_tl = detections[0]

    scene_color = best_tl["color"]

    # ===== ACTION MAPPING =====
    if scene_color == "red":
        action = "Stop"
    elif scene_color == "yellow":
        action = "Slow"
    elif scene_color == "green":
        action = "Go"
    else:
        action = "Unknown"

    print("\nDetections:")
    for d in detections:
        print(d)

    print("\nScene Traffic Light Color:", scene_color)
    print("Vehicle Action:", action)

    return {
        "scene_color": scene_color,
        "action": action,
        "detections": detections
    }

In [22]:
os.path.exists("data/bdd_100k_val/images/b4542860-0b880bb4.jpg")

True

In [23]:
# =========================
# RUN
# =========================

test_image = "data/bdd_100k_val/images/b4542860-0b880bb4.jpg"
infer_image(test_image)


image 1/1 c:\Users\kaleem\Learn\FUB\WiSe_2025_26\Knowledge Graphs and AI Apps\tutorials\project\images2action\data\bdd_100k_val\images\b4542860-0b880bb4.jpg: 384x640 1 traffic_light, 106.5ms
Speed: 5.1ms preprocess, 106.5ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

Detections:
{'box': (np.float32(646.1471), np.float32(254.28204), np.float32(657.5294), np.float32(279.64984)), 'color': 'green', 'score': np.float32(0.6872729)}

Scene Traffic Light Color: green
Vehicle Action: Go


{'scene_color': 'green',
 'action': 'Go',
 'detections': [{'box': (np.float32(646.1471),
    np.float32(254.28204),
    np.float32(657.5294),
    np.float32(279.64984)),
   'color': 'green',
   'score': np.float32(0.6872729)}]}